In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.appName("ParquetDeltaExample").getOrCreate()
data = [
    (1, "Alice", "IT", 60000),
    (2, "Bob", "HR", 40000),
    (3, "Charlie", "IT", 80000),
    (4, "David", "Finance", 30000)
]
columns = ["id", "name", "department", "salary"]
df = spark.createDataFrame(data, columns)
df.write.mode("overwrite").parquet("/FileStore/parquet/employees")

In [0]:
parquet_df=spark.read.parquet("/FileStore/parquet/employees")
parquet_df.show()

+---+-------+----------+------+
| id|   name|department|salary|
+---+-------+----------+------+
|  4|  David|   Finance| 30000|
|  3|Charlie|        IT| 80000|
|  1|  Alice|        IT| 60000|
|  2|    Bob|        HR| 40000|
+---+-------+----------+------+



In [0]:
high_value_df=parquet_df.filter(col("salary")>50000)
high_value_df.show()

+---+-------+----------+------+
| id|   name|department|salary|
+---+-------+----------+------+
|  3|Charlie|        IT| 80000|
|  1|  Alice|        IT| 60000|
+---+-------+----------+------+



In [0]:
df.write.mode("overwrite").partitionBy("department").parquet("/FileStore/parquet/employees_partitioned")

In [0]:
it_df=spark.read.parquet("/FileStore/parquet/employees_partitioned/department=IT")
it_df.show()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  3|Charlie| 80000|
|  1|  Alice| 60000|
+---+-------+------+



In [0]:
new_data=[(5,"Emma","HR",70000)]
new_df=spark.createDataFrame(new_data,columns)
new_df.write.mode("append").parquet("/FileStore/parquet/employees")

In [0]:
parquet_df=spark.read.parquet("/FileStore/parquet/employees")
parquet_df.createOrReplaceTempView("employee_parquet")
spark.sql("SELECT * FROM employee_parquet").show()





+---+-------+----------+------+
| id|   name|department|salary|
+---+-------+----------+------+
|  4|  David|   Finance| 30000|
|  3|Charlie|        IT| 80000|
|  1|  Alice|        IT| 60000|
|  2|    Bob|        HR| 40000|
+---+-------+----------+------+



In [0]:
parquet_data=spark.read.parquet("/FileStore/parquet/employees")
parquet_data.write.format("delta").mode("overwrite").save("/FileStore/delta/employees_delta")

In [0]:
from delta.tables import DeltaTable
delta_table = DeltaTable.forPath(spark, "/FileStore/delta/employees_delta")
delta_table.update(condition="id = 2",set={"salary": "45000"})
delta_table.toDF().show()

+---+-------+----------+------+
| id|   name|department|salary|
+---+-------+----------+------+
|  1|  Alice|        IT| 60000|
|  3|Charlie|        IT| 80000|
|  4|  David|   Finance| 30000|
|  2|    Bob|        HR| 45000|
+---+-------+----------+------+



Parquet:
Parquet is a columnar storage file format.
It is mainly used for storing large datasets efficiently.
Provides good compression and faster read performance.
Best for read-heavy analytics workloads.
Does not support direct UPDATE or DELETE operations.
No ACID transaction support.
No version history or time travel.
Suitable for simple data lakes and reporting.


Delta Lake:
Delta is a storage layer built on top of Parquet.
Stores data in Parquet format with an added transaction log (_delta_log).
Supports ACID transactions for reliability.
Allows UPDATE, DELETE, and MERGE operations.
Provides schema enforcement and schema evolution.
Supports time travel (access old versions of data).
Prevents data corruption during failed writes.
Better for production pipelines and frequently changing data.